In [2]:
import pandas as pd
import numpy as np
import time
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs('../results/smote_experiments', exist_ok=True)
os.makedirs('../results/smote_experiments/figures', exist_ok=True)

LABEL_MAP    = {0: 'Benign', 1: 'DoS', 2: 'DDoS', 3: 'Mirai', 4: 'Spoofing'}
CLASS_NAMES  = ['Benign', 'DoS', 'DDoS', 'Mirai', 'Spoofing']
ATTACK_NAMES = ['DoS', 'DDoS', 'Mirai', 'Spoofing']
MODEL_NAMES  = ['DT', 'RF', 'XGB', 'LGBM', 'CAT', 'BRF']

print("✓ Imports done.")

✓ Imports done.


In [3]:
X_train = pd.read_csv('../data/X_train.csv')
X_test  = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/y_test.csv').squeeze()

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"\nClass distribution (train):\n{y_train.value_counts().sort_index().rename(LABEL_MAP)}")

X_train: (1062114, 30)
X_test:  (265529, 30)

Class distribution (train):
Label_encoded
Benign      318298
DoS         239759
DDoS        239095
Mirai       139403
Spoofing    125559
Name: count, dtype: int64


In [4]:
# Binary labels
y_train_binary = y_train.apply(lambda x: 'Benign' if x == 0 else 'Attack')
y_test_binary  = y_test.apply(lambda x: 'Benign' if x == 0 else 'Attack')

le_binary = LabelEncoder()
y_train_binary_enc = le_binary.fit_transform(y_train_binary)
y_test_binary_enc  = le_binary.transform(y_test_binary)
print(f"Binary encoding: {dict(zip(le_binary.classes_, le_binary.transform(le_binary.classes_)))}")

# Attack-only rows
attack_mask_train = y_train != 0
attack_mask_test  = y_test != 0
X_train_attack    = X_train[attack_mask_train]
y_train_attack    = y_train[attack_mask_train]
X_test_attack     = X_test[attack_mask_test]
y_test_attack     = y_test[attack_mask_test]

le_attack = LabelEncoder()
y_train_attack_enc = le_attack.fit_transform(y_train_attack)
y_test_attack_enc  = le_attack.transform(y_test_attack)
print(f"Attack encoding: {dict(zip(le_attack.classes_, le_attack.transform(le_attack.classes_)))}")
print(f"\nStage 2 distribution (train):\n{y_train_attack.value_counts().sort_index().rename(LABEL_MAP)}")

Binary encoding: {'Attack': np.int64(0), 'Benign': np.int64(1)}
Attack encoding: {np.int64(1): np.int64(0), np.int64(2): np.int64(1), np.int64(3): np.int64(2), np.int64(4): np.int64(3)}

Stage 2 distribution (train):
Label_encoded
DoS         239759
DDoS        239095
Mirai       139403
Spoofing    125559
Name: count, dtype: int64


In [5]:
def get_models():
    return {
        'DT':   DecisionTreeClassifier(random_state=42),
        'RF':   RandomForestClassifier(random_state=42, n_jobs=-1),
        'XGB':  XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss'),
        'LGBM': LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
        'CAT':  CatBoostClassifier(random_state=42, verbose=0),
        'BRF':  BalancedRandomForestClassifier(random_state=42, n_jobs=-1,
                                                n_estimators=50, max_depth=20)
    }

def get_models_multiclass():
    return {
        'DT':   DecisionTreeClassifier(random_state=42),
        'RF':   RandomForestClassifier(random_state=42, n_jobs=-1),
        'XGB':  XGBClassifier(random_state=42, n_jobs=-1, eval_metric='mlogloss'),
        'LGBM': LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
        'CAT':  CatBoostClassifier(random_state=42, verbose=0),
        'BRF':  BalancedRandomForestClassifier(random_state=42, n_jobs=-1,
                                                n_estimators=50, max_depth=20)
    }


def evaluate_stage1(y_true_binary, y_pred_binary, y_true_multiclass):
    acc  = accuracy_score(y_true_binary, y_pred_binary)
    prec = precision_score(y_true_binary, y_pred_binary, pos_label='Attack')
    rec  = recall_score(y_true_binary, y_pred_binary, pos_label='Attack')
    f1   = f1_score(y_true_binary, y_pred_binary, pos_label='Attack')
    cm   = confusion_matrix(y_true_binary, y_pred_binary, labels=['Benign', 'Attack'])
    fnr  = cm[1][0] / (cm[1][0] + cm[1][1])
    fpr  = cm[0][1] / (cm[0][0] + cm[0][1])

    per_class_fnr = {}
    y_pred_arr = np.array(y_pred_binary)
    for label, name in LABEL_MAP.items():
        if label == 0:
            continue
        mask   = y_true_multiclass == label
        missed = np.sum(y_pred_arr[mask] == 'Benign')
        total  = mask.sum()
        per_class_fnr[name] = missed / total if total > 0 else 0

    return {
        'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
        'fnr': fnr, 'fpr': fpr, 'per_class_fnr': per_class_fnr,
        'confusion_matrix': cm.tolist()
    }


def evaluate_stage2(y_true, y_pred, le=None):
    labels = sorted(np.unique(y_true))
    acc    = accuracy_score(y_true, y_pred)
    prec   = precision_score(y_true, y_pred, average='macro')
    rec    = recall_score(y_true, y_pred, average='macro')
    f1     = f1_score(y_true, y_pred, average='macro')

    per_class = {}
    for l in labels:
        tp = np.sum((y_true == l) & (y_pred == l))
        fp = np.sum((y_true != l) & (y_pred == l))
        fn = np.sum((y_true == l) & (y_pred != l))
        p  = tp / (tp + fp) if (tp + fp) > 0 else 0
        r  = tp / (tp + fn) if (tp + fn) > 0 else 0
        f  = 2 * p * r / (p + r) if (p + r) > 0 else 0
        per_class[LABEL_MAP[l]] = {'precision': p, 'recall': r, 'f1': f}

    return {
        'accuracy': acc, 'precision_macro': prec,
        'recall_macro': rec, 'f1_macro': f1,
        'per_class': per_class
    }


def evaluate_pipeline(y_true, y_final_pred):
    all_labels  = sorted(LABEL_MAP.keys())
    acc         = accuracy_score(y_true, y_final_pred)
    f1_macro    = f1_score(y_true, y_final_pred, average='macro')
    f1_weighted = f1_score(y_true, y_final_pred, average='weighted')
    prec_macro  = precision_score(y_true, y_final_pred, average='macro')
    rec_macro   = recall_score(y_true, y_final_pred, average='macro')
    cm          = confusion_matrix(y_true, y_final_pred, labels=all_labels)
    fnr         = np.sum((y_true != 0) & (y_final_pred == 0)) / np.sum(y_true != 0)
    fpr         = np.sum((y_true == 0) & (y_final_pred != 0)) / np.sum(y_true == 0)

    return {
        'accuracy': acc, 'precision_macro': prec_macro,
        'recall_macro': rec_macro, 'f1_macro': f1_macro,
        'f1_weighted': f1_weighted, 'fnr': fnr, 'fpr': fpr,
        'confusion_matrix': cm.tolist()
    }


def run_flat(name, model, X_tr, y_tr, le_flat):
    """Train and evaluate a flat 5-class model."""
    print(f"  Flat {name}...", end=' ', flush=True)
    t0 = time.time()
    model.fit(X_tr, le_flat.transform(y_tr))
    train_time = time.time() - t0

    y_pred_enc = model.predict(X_test)
    y_pred     = le_flat.inverse_transform(y_pred_enc)

    acc         = accuracy_score(y_test, y_pred)
    f1_macro    = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    prec_macro  = precision_score(y_test, y_pred, average='macro')
    rec_macro   = recall_score(y_test, y_pred, average='macro')
    cm          = confusion_matrix(y_test, y_pred,
                                    labels=sorted(LABEL_MAP.keys()))
    fnr = np.sum((y_test != 0) & (y_pred == 0)) / np.sum(y_test != 0)
    fpr = np.sum((y_test == 0) & (y_pred != 0)) / np.sum(y_test == 0)

    per_class = {}
    for l in sorted(LABEL_MAP.keys()):
        tp = np.sum((y_test == l) & (y_pred == l))
        fp = np.sum((y_test != l) & (y_pred == l))
        fn = np.sum((y_test == l) & (y_pred != l))
        p  = tp / (tp + fp) if (tp + fp) > 0 else 0
        r  = tp / (tp + fn) if (tp + fn) > 0 else 0
        f  = 2 * p * r / (p + r) if (p + r) > 0 else 0
        per_class[LABEL_MAP[l]] = {'precision': p, 'recall': r, 'f1': f}

    print(f"Acc: {acc:.4f} | F1: {f1_macro:.4f} | FNR: {fnr:.4f} ({train_time:.1f}s)")
    return {
        'accuracy': acc, 'precision_macro': prec_macro,
        'recall_macro': rec_macro, 'f1_macro': f1_macro,
        'f1_weighted': f1_weighted, 'fnr': fnr, 'fpr': fpr,
        'per_class': per_class, 'confusion_matrix': cm.tolist(),
        'train_time': train_time
    }


def run_hierarchical(name, s1_model, s2_model,
                     X_tr_s1, y_tr_s1_enc,
                     X_tr_s2, y_tr_s2_enc,
                     le_s2=None):
    """Train Stage 1 + Stage 2, combine, return all metrics."""
    if le_s2 is None:
        le_s2 = le_attack

    print(f"  Hier {name}...")

    # Stage 1
    t0 = time.time()
    s1_model.fit(X_tr_s1, y_tr_s1_enc)
    s1_time = time.time() - t0
    s1_pred_enc = s1_model.predict(X_test)
    s1_pred     = le_binary.inverse_transform(s1_pred_enc)
    s1_metrics  = evaluate_stage1(y_test_binary, s1_pred, y_test.values)

    # Stage 2
    t0 = time.time()
    s2_model.fit(X_tr_s2, y_tr_s2_enc)
    s2_time = time.time() - t0
    s2_pred_enc = s2_model.predict(X_test_attack)
    s2_pred     = le_s2.inverse_transform(s2_pred_enc)
    s2_metrics  = evaluate_stage2(y_test_attack.values, s2_pred)

    # Pipeline
    final_pred = np.empty(len(y_test), dtype=object)
    benign_idx = np.where(s1_pred == 'Benign')[0]
    attack_idx = np.where(s1_pred == 'Attack')[0]
    final_pred[benign_idx] = 0
    if len(attack_idx) > 0:
        enc = s2_model.predict(X_test.iloc[attack_idx])
        final_pred[attack_idx] = le_s2.inverse_transform(enc)
    final_pred   = final_pred.astype(int)
    pipe_metrics = evaluate_pipeline(y_test.values, final_pred)

    print(f"    S1 FNR: {s1_metrics['fnr']:.4f} | "
          f"S2 F1: {s2_metrics['f1_macro']:.4f} | "
          f"Pipe Acc: {pipe_metrics['accuracy']:.4f} | "
          f"Pipe F1: {pipe_metrics['f1_macro']:.4f} "
          f"({s1_time:.1f}s + {s2_time:.1f}s)")

    return {
        'stage1': s1_metrics,
        'stage2': s2_metrics,
        'pipeline': pipe_metrics
    }


def make_json_safe(d):
    if isinstance(d, dict):  return {k: make_json_safe(v) for k, v in d.items()}
    if isinstance(d, list):  return [make_json_safe(v) for v in d]
    if isinstance(d, (np.integer,)):  return int(d)
    if isinstance(d, (np.floating,)): return float(d)
    if d is None: return None
    return d


print("✓ Helper functions ready.")

✓ Helper functions ready.


In [ ]:

# ── H1: Binary SMOTE after merging ───────────────────────────────────────────
print("\nPreparing H1: Binary SMOTE after merging...")
smote_binary = SMOTE(random_state=42, k_neighbors=5)
X_train_h1, y_train_h1_enc = smote_binary.fit_resample(X_train, y_train_binary_enc)
print(f"H1 distribution: {pd.Series(y_train_h1_enc).value_counts().to_dict()}")

# ── H2/H4: Subclass-aware SMOTE (Mirai + Spoofing → 300,000) ─────────────────
print("\nPreparing H2/H4: Subclass-aware SMOTE...")
target_count      = 300000
sampling_strategy = {
    label: target_count
    for label in [3, 4]  # Mirai=3, Spoofing=4
    if (y_train == label).sum() < target_count
}
smote_subclass = SMOTE(random_state=42, k_neighbors=5,
                        sampling_strategy=sampling_strategy)
X_train_h2_5class, y_train_h2_5class = smote_subclass.fit_resample(
    X_train, y_train
)
print(f"H2/H4 5-class distribution:\n"
      f"{pd.Series(y_train_h2_5class).value_counts().sort_index().rename(LABEL_MAP)}")

# Collapse to binary for Stage 1
y_train_h2_binary     = pd.Series(y_train_h2_5class).apply(
    lambda x: 'Benign' if x == 0 else 'Attack'
)
y_train_h2_binary_enc = le_binary.transform(y_train_h2_binary)

# Attack-only subset from H2 data (for H4 Stage 2)
attack_mask_h2    = np.array(y_train_h2_5class) != 0
X_train_h2_attack = X_train_h2_5class[attack_mask_h2]
y_train_h2_attack = np.array(y_train_h2_5class)[attack_mask_h2]

le_attack_h2         = LabelEncoder()
y_train_h2_attack_enc = le_attack_h2.fit_transform(y_train_h2_attack)

# ── H3/H4: SMOTE on attack-only Stage 2 data ─────────────────────────────────
print("\nPreparing H3: SMOTE on attack-only Stage 2 data...")
smote_s2 = SMOTE(random_state=42, k_neighbors=5)
X_train_h3_s2, y_train_h3_s2 = smote_s2.fit_resample(
    X_train_attack, y_train_attack_enc
)
print(f"H3 S2 distribution: {pd.Series(y_train_h3_s2).value_counts().to_dict()}")

# H4 Stage 2: SMOTE on H2 attack-only data
print("\nPreparing H4 Stage 2: SMOTE on subclass-aware attack data...")
smote_h4_s2 = SMOTE(random_state=42, k_neighbors=5)
X_train_h4_s2, y_train_h4_s2 = smote_h4_s2.fit_resample(
    X_train_h2_attack, y_train_h2_attack_enc
)
print(f"H4 S2 distribution: {pd.Series(y_train_h4_s2).value_counts().to_dict()}")

print("\n✓ All SMOTE variants ready.")

Preparing F1: Global 5-class SMOTE...


NameError: name 'y_train_enc' is not defined

In [ ]:
print("\n" + "="*60)
print("H1: BINARY SMOTE S1 + SAME MODEL S2 (no SMOTE)")
print("="*60)

H1 = {}
for name in MODEL_NAMES:
    models = get_models()
    s1_m   = get_models()[name]
    s2_m   = get_models_multiclass()[name]
    H1[name] = run_hierarchical(
        name,
        s1_model=s1_m,
        s2_model=s2_m,
        X_tr_s1=X_train_h1,
        y_tr_s1_enc=y_train_h1_enc,
        X_tr_s2=X_train_attack,
        y_tr_s2_enc=y_train_attack_enc
    )

print("✓ H1 done.")

In [ ]:
print("\n" + "="*60)
print("H2: SUBCLASS-AWARE SMOTE S1 + SAME MODEL S2 (no SMOTE)")
print("="*60)

H2 = {}
for name in MODEL_NAMES:
    s1_m = get_models()[name]
    s2_m = get_models_multiclass()[name]
    H2[name] = run_hierarchical(
        name,
        s1_model=s1_m,
        s2_model=s2_m,
        X_tr_s1=X_train_h2_5class,
        y_tr_s1_enc=y_train_h2_binary_enc,
        X_tr_s2=X_train_attack,
        y_tr_s2_enc=y_train_attack_enc
    )

print("✓ H2 done.")

In [ ]:
print("\n" + "="*60)
print("H3: RF S1 (no SMOTE) + SMOTE on ATTACK-ONLY S2")
print("="*60)

H3 = {}
for name in MODEL_NAMES:
    s1_m = get_models()[name]
    s2_m = get_models_multiclass()[name]
    H3[name] = run_hierarchical(
        name,
        s1_model=s1_m,
        s2_model=s2_m,
        X_tr_s1=X_train,
        y_tr_s1_enc=y_train_binary_enc,
        X_tr_s2=X_train_h3_s2,
        y_tr_s2_enc=y_train_h3_s2
    )

print("✓ H3 done.")

In [ ]:
print("\n" + "="*60)
print("H4: SUBCLASS-AWARE SMOTE S1 + SMOTE S2")
print("="*60)

H4 = {}
for name in MODEL_NAMES:
    s1_m = get_models()[name]
    s2_m = get_models_multiclass()[name]
    H4[name] = run_hierarchical(
        name,
        s1_model=s1_m,
        s2_model=s2_m,
        X_tr_s1=X_train_h2_5class,
        y_tr_s1_enc=y_train_h2_binary_enc,
        X_tr_s2=X_train_h4_s2,
        y_tr_s2_enc=y_train_h4_s2,
        le_s2=le_attack_h2
    )

print("✓ H4 done.")

In [ ]:
experiments_hier = {'H1': H1, 'H2': H2, 'H3': H3, 'H4': H4}

# ── Table 1: Full pipeline comparison ───────────────────────────────────────
print("\n" + "="*80)
print("TABLE 1: FULL PIPELINE COMPARISON")
print("="*80)
print(f"{'Config':<12} {'Model':<8} {'Accuracy':>10} {'Macro F1':>10} "
      f"{'Weighted F1':>12} {'FNR':>8} {'FPR':>8}")
print("-"*80)

for exp_name, exp in experiments_hier.items():
    for model_name in MODEL_NAMES:
        r = exp[model_name]['pipeline']
        print(f"{exp_name:<12} {model_name:<8} "
              f"{r['accuracy']:>10.4f} {r['f1_macro']:>10.4f} "
              f"{r['f1_weighted']:>12.4f} {r['fnr']:>8.4f} {r['fpr']:>8.4f}")
    print()

# ── Table 2: Stage 1 per-class FNR ──────────────────────────────────────────
print("\n" + "="*80)
print("TABLE 2: STAGE 1 PER-ATTACK-CLASS FNR (hierarchical only)")
print("="*80)
print(f"{'Config':<12} {'Model':<8} {'DoS':>8} {'DDoS':>8} "
      f"{'Mirai':>8} {'Spoofing':>10} {'Overall FNR':>12}")
print("-"*80)

for exp_name, exp in experiments_hier.items():
    for model_name in MODEL_NAMES:
        pcf = exp[model_name]['stage1']['per_class_fnr']
        fnr = exp[model_name]['stage1']['fnr']
        print(f"{exp_name:<12} {model_name:<8} "
              f"{pcf.get('DoS', 0):>8.4f} {pcf.get('DDoS', 0):>8.4f} "
              f"{pcf.get('Mirai', 0):>8.4f} {pcf.get('Spoofing', 0):>10.4f} "
              f"{fnr:>12.4f}")
    print()

# ── Table 3: Stage 2 per-class F1 ───────────────────────────────────────────
print("\n" + "="*80)
print("TABLE 3: STAGE 2 PER-CLASS F1 (hierarchical only)")
print("="*80)
print(f"{'Config':<12} {'Model':<8} {'DoS':>8} {'DDoS':>8} "
      f"{'Mirai':>8} {'Spoofing':>10} {'Macro F1':>10}")
print("-"*80)

for exp_name, exp in experiments_hier.items():
    for model_name in MODEL_NAMES:
        pc  = exp[model_name]['stage2']['per_class']
        mf1 = exp[model_name]['stage2']['f1_macro']
        print(f"{exp_name:<12} {model_name:<8} "
              f"{pc['DoS']['f1']:>8.4f} {pc['DDoS']['f1']:>8.4f} "
              f"{pc['Mirai']['f1']:>8.4f} {pc['Spoofing']['f1']:>10.4f} "
              f"{mf1:>10.4f}")
    print()



NameError: name 'H1' is not defined

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

config_styles = {
    'H1': ('^', '#3498db'),
    'H2': ('D', '#e74c3c'),
    'H3': ('P', '#2ecc71'),
    'H4': ('*', '#9b59b6'),
}

model_colors = {
    'DT': '#e74c3c', 'RF': '#3498db', 'XGB': '#2ecc71',
    'LGBM': '#f39c12', 'CAT': '#9b59b6', 'BRF': '#1abc9c'
}

for exp_name, exp in experiments_hier.items():
    marker, color = config_styles[exp_name]
    for model_name in MODEL_NAMES:
        r   = exp[model_name]['pipeline']
        fnr = r['fnr']
        f1  = r['f1_macro']
        ax.scatter(fnr, f1, marker=marker,
                   color=model_colors[model_name], s=150, zorder=5)

from matplotlib.lines import Line2D
config_legend = [Line2D([0], [0], marker=v[0], color='gray', linestyle='None',
                         markersize=8, label=k)
                 for k, v in config_styles.items()]
model_legend  = [Line2D([0], [0], marker='o', color=c, linestyle='None',
                          markersize=8, label=m)
                 for m, c in model_colors.items()]

l1 = ax.legend(handles=config_legend, title='Config',
               loc='upper right', fontsize=9)
ax.add_artist(l1)
ax.legend(handles=model_legend, title='Model',
          loc='lower left', fontsize=9)

ax.set_xlabel('Attack FNR ↓ better', fontsize=11)
ax.set_ylabel('Macro F1 ↑ better', fontsize=11)
ax.set_title('FNR vs Macro F1 — H1/H2/H3/H4 × All Models\n(ideal = top-left)',
             fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/smote_experiments/figures/fnr_vs_f1_all.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fnr_vs_f1_all.png")

In [ ]:
all_results = {
    'H1': make_json_safe(H1),
    'H2': make_json_safe(H2),
    'H3': make_json_safe(H3),
    'H4': make_json_safe(H4),
}

with open('../results/smote_experiments/all_smote_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print("✓ Saved to ../results/smote_experiments/all_smote_results.json")
print("✓ Figure saved to ../results/smote_experiments/figures/fnr_vs_f1_all.png")